<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 3 - Data Diagnosis**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/PVAR_japan_endogenous_money

### **1) REQUIREMENTS SET-UP**

In [1]:
# Requirements.txt file installation
# !pip install -r requirements.txt

In [2]:
# Libraries import
import warnings
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
import matplotlib.dates as mdates
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats import levene
from scipy.stats import ks_2samp
from scipy.stats import kstest
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.ar_model import AutoReg
from statsmodels.tsa.filters.hp_filter import hpfilter
import sklearn.tree
import sklearn.metrics
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing 
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             accuracy_score, precision_recall_curve, auc, 
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import (LinearRegression, LogisticRegression)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px
import openpyxl as pxl
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML
from IPython.display import Image
import itertools
from arch.unitroot import PhillipsPerron

In [3]:
# Aggregated (raw) and Transformed df import 
jp_aggregated_df = pd.read_csv("Data/Aggregated/jp_aggregated_df.csv")
jp_trans_df = pd.read_csv("Data/Transformed/jp_trans_df.csv")

In [4]:
# Metrics clusters for plotting
metrics_group = {
    "Monetary Aggregates" : ["Monetary Aggregates - M1 (JPY)", "Monetary Aggregates - M2 (JPY)", "Monetary Aggregates - M3 (JPY)"],
    "Credit Metrics" : ["Total Credit - General Government (%GDP)", "Total Credit - Households & NPISHs (%GDP)", "Total Credit - Private Non-Financial (%GDP)"],
    "Reserves" : ["Total Treasury Reserves (- Gold)"],
    "Monetary Policy Proxies (Yields)" : ["10-Year Gov Bond Yields (%)", "Call Money/Interbank Immediate (%)", "Est. 1-year Neutral Interest Rate (%)", "Est. 10-year Neutral Interest Rate (%)"], 
    "Exchange Rate" : ["USD-JPY reer CPI-based (Index 2015=100)", "JPY-USD Spot Exchange Rate"],
    "Output-Trends": ["Real GDP (billions chained 2015 JPY)"],
    "Consumption Prices": ["HICP (SA)"],
    "Debt Metrics" : ["Central Government Debt (% GDP)", "Domestic Private Debt Securities (% GDP)", "Domestic Public Debt Securities (% GDP)"],
    "Banking Sector" : ["Loan Interest Rate (%)", "Deposit Interest Rate (%)", "1615.T-Price"],
    "Controls" : ["10-Year US T-Bills Yield (%)", "CBOE-VIX"],
    "BoJ Total Assets" : ["BoJ’s Total Assets (100 Million Yen)"]
}

### **2) AGGREGATED (RAW) DATA STATIONARITY TESTING**

In [5]:
# Autocorrelation coefficients AR(1)
# Drop non-numeric columns and rows with missing values
df = jp_aggregated_df.copy()
jp_aggregated_numeric = df.drop(columns=["Country", "Time"]).dropna()

# AR(1) autocorrelation for each variable
ar1_results = {}
for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col]

# (!!!) lag-1 autocorrelation
    ar1 = series.autocorr(lag=1)
    ar1_results[col] = ar1

# Better to create a dataframe to display the results
jp_ar1_df = pd.DataFrame.from_dict(ar1_results, orient="index", columns=["AR(1)"])
jp_ar1_df

,AR(1)
Monetary Aggregates - M1 (JPY),0.999761
Monetary Aggregates - M2 (JPY),0.999878
Monetary Aggregates - M3 (JPY),0.999875
Total Credit - Private Non-Financial (%GDP),0.985710
Total Credit - General Government (%GDP),0.997071
Total Credit - Households & NPISHs (%GDP),0.982341
Total Treasury Reserves (- Gold),0.990202
10-Year Gov Bond Yields (%),0.983519
Call Money/Interbank Immediate (%),0.985473
Est. 1-year Neutral Interest Rate (%),0.985248


In [6]:
# Unit-root Testing - Adfuller Test 
# Drop non-numeric columns and handle missing data
df = jp_aggregated_df.copy()
jp_aggregated_numeric = df.drop(columns=["Country", "Time"]).dropna()

# (!!!) We need to initialize the results as empty list before execuding the test
results = []

for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col]

# As before, we extract the AR(1) coefficients
    ar1 = series.autocorr(lag=1)

# Augmented Dickey-Fuller (ADF) unit root test 
    adf_result = adfuller(series, autolag="AIC")
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    crit_values = adf_result[4]

    results.append({
        "Variable": col,
        "ADF Statistic": adf_stat,
        "p-value": p_value,
        "Stationary - Absence of unit-root (HP1)": "Yes" if p_value < 0.05 else "No"
    })

jp_adf_df = pd.DataFrame(results)
jp_adf_df

,Variable,ADF Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,Monetary Aggregates - M1 (JPY),2.145042,0.998834,No
1,Monetary Aggregates - M2 (JPY),7.421314,1.000000,No
2,Monetary Aggregates - M3 (JPY),6.495981,1.000000,No
3,Total Credit - Private Non-Financial (%GDP),-1.273887,0.641033,No
4,Total Credit - General Government (%GDP),-3.378474,0.011715,Yes
5,Total Credit - Households & NPISHs (%GDP),-0.939533,0.774675,No
6,Total Treasury Reserves (- Gold),-1.598362,0.484416,No
7,10-Year Gov Bond Yields (%),-0.734868,0.837510,No
8,Call Money/Interbank Immediate (%),-3.532602,0.007188,Yes
9,Est. 1-year Neutral Interest Rate (%),-1.543562,0.511863,No


In [7]:
# Unit-root Testing - Phillips-Perron Test 
# (!!!) We need to initialize the results as empty list before execuding the test
pp_results = []

for col in jp_aggregated_numeric.columns:
    series = jp_aggregated_numeric[col].dropna()
    
# Phillips–Perron test 
# (!!!) From arch instead of stats.models is much smoother
    test = PhillipsPerron(series)
    pp_results.append({
        "Variable": col,
        "PP Statistic": test.stat,
        "p-value": test.pvalue,
        "Stationary - Absence of unit-root (HP1)": "Yes" if test.pvalue < 0.05 else "No"
    })

jp_pp_df = pd.DataFrame(pp_results)
jp_pp_df

,Variable,PP Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,Monetary Aggregates - M1 (JPY),5.499319,1.000000,No
1,Monetary Aggregates - M2 (JPY),15.311005,1.000000,No
2,Monetary Aggregates - M3 (JPY),11.710651,1.000000,No
3,Total Credit - Private Non-Financial (%GDP),-1.103513,0.713669,No
4,Total Credit - General Government (%GDP),-2.459606,0.125614,No
5,Total Credit - Households & NPISHs (%GDP),-1.420685,0.572320,No
6,Total Treasury Reserves (- Gold),-1.615145,0.475275,No
7,10-Year Gov Bond Yields (%),-0.282190,0.927989,No
8,Call Money/Interbank Immediate (%),-2.927652,0.042226,Yes
9,Est. 1-year Neutral Interest Rate (%),-1.479774,0.543492,No


### **3) TRANSFORMED DATA STATIONARITY RE-TESTING**

In [8]:
# Autocorrelation coefficients AR(1)
# Drop non-numeric columns and rows with missing values
df = jp_trans_df.copy()
jp_transformed_numeric = df.drop(columns=["Country", "Time"]).dropna()

# AR(1) autocorrelation for each variable
ar1_results = {}
for col in jp_transformed_numeric.columns:
    series = jp_transformed_numeric[col]

# (!!!) lag-1 autocorrelation
    ar1 = series.autocorr(lag=1)
    ar1_results[col] = ar1

# Better to create a dataframe to display the results
jp_ar1_df = pd.DataFrame.from_dict(ar1_results, orient="index", columns=["AR(1)"])
jp_ar1_df

,AR(1)
LogDiff-Monetary Aggregates - M1 (JPY),0.345322
LogDiff-Monetary Aggregates - M2 (JPY),0.166549
LogDiff-Monetary Aggregates - M3 (JPY),0.266801
LogDiff-Total Treasury Reserves (- Gold),-0.011823
LogDiff-USD-JPY reer CPI-based (Index 2015=100),0.360090
LogDiff-JPY-USD Spot Exchange Rate,0.317363
LogDiff-HICP (SA),0.302991
LogDiff-1615.T-Price,0.083562
LogDiff-BoJ’s Total Assets (100 Million Yen),-0.234426
LogDiff-Call Money/Interbank Immediate (%),0.163523


In [11]:
# Unit-root Testing - Adfuller Test 
# Drop non-numeric columns and handle missing data
df = jp_trans_df.copy()
jp_trans_numeric = df.drop(columns=["Country", "Time"]).dropna()

# (!!!) We need to initialize the results as empty list before execuding the test
results = []

for col in jp_trans_numeric.columns:
    series = jp_trans_numeric[col]

# As before, we extract the AR(1) coefficients
    ar1 = series.autocorr(lag=1)

# Augmented Dickey-Fuller (ADF) unit root test 
    adf_result = adfuller(series, autolag="AIC")
    adf_stat = adf_result[0]
    p_value = adf_result[1]
    crit_values = adf_result[4]

    results.append({
        "Variable": col,
        "ADF Statistic": adf_stat,
        "p-value": p_value,
        "Stationary - Absence of unit-root (HP1)": "Yes" if p_value < 0.05 else "No"
    })

jp_adf_df = pd.DataFrame(results)
jp_adf_df

,Variable,ADF Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,LogDiff-Monetary Aggregates - M1 (JPY),-2.444472,1.295758e-01,No
1,LogDiff-Monetary Aggregates - M2 (JPY),-8.189657,7.729193e-13,Yes
2,LogDiff-Monetary Aggregates - M3 (JPY),-7.422513,6.678746e-11,Yes
3,LogDiff-Total Treasury Reserves (- Gold),-4.323465,4.044364e-04,Yes
4,LogDiff-USD-JPY reer CPI-based (Index 2015=100),-6.641602,5.392505e-09,Yes
5,LogDiff-JPY-USD Spot Exchange Rate,-6.896973,1.311311e-09,Yes
6,LogDiff-HICP (SA),-7.072277,4.902224e-10,Yes
7,LogDiff-1615.T-Price,-8.614095,6.359273e-14,Yes
8,LogDiff-BoJ’s Total Assets (100 Million Yen),-1.239708,6.562852e-01,No
9,LogDiff-Call Money/Interbank Immediate (%),-6.230479,4.967541e-08,Yes


In [10]:
# Unit-root Testing - Phillips-Perron Test 
# (!!!) We need to initialize the results as empty list before execuding the test
pp_results = []

for col in jp_trans_numeric.columns:
    series = jp_trans_numeric[col].dropna()
    
# Phillips–Perron test 
# (!!!) From arch instead of stats.models is much smoother
    test = PhillipsPerron(series)
    pp_results.append({
        "Variable": col,
        "PP Statistic": test.stat,
        "p-value": test.pvalue,
        "Stationary - Absence of unit-root (HP1)": "Yes" if test.pvalue < 0.05 else "No"
    })

jp_pp_df = pd.DataFrame(pp_results)
jp_pp_df

,Variable,PP Statistic,p-value,Stationary - Absence of unit-root (HP1)
0,LogDiff-Monetary Aggregates - M1 (JPY),-7.835884,6.117131e-12,Yes
1,LogDiff-Monetary Aggregates - M2 (JPY),-8.292218,4.231668e-13,Yes
2,LogDiff-Monetary Aggregates - M3 (JPY),-7.608306,2.290720e-11,Yes
3,LogDiff-Total Treasury Reserves (- Gold),-9.845106,4.642030e-17,Yes
4,LogDiff-USD-JPY reer CPI-based (Index 2015=100),-6.364062,2.434323e-08,Yes
5,LogDiff-JPY-USD Spot Exchange Rate,-6.926089,1.114397e-09,Yes
6,LogDiff-HICP (SA),-7.211023,2.234738e-10,Yes
7,LogDiff-1615.T-Price,-8.633066,5.686364e-14,Yes
8,LogDiff-BoJ’s Total Assets (100 Million Yen),-12.009884,3.199194e-22,Yes
9,LogDiff-Call Money/Interbank Immediate (%),-6.218345,5.297819e-08,Yes
